# EDA v3 — Cat 1050 (Phòng Trọ)

**Quy trình:**
1. **Giai đoạn 1 (Zone 0-7)** — Tiền xử lý đầy đủ 5 nhóm issue A-E của Cursor.md (TEXT only, không chart).
2. **Giai đoạn 2 (Zone 8-12)** — EDA có chart + insight nghiệp vụ + đề xuất 2 hướng giải pháp.

**Đã xác nhận:** Cat 1050 = Phòng trọ (Cursor.md override README). User scope: login + non-login (session-scoped).

## Zone 0 — Setup & Imports

In [ ]:
# 0.1 — Imports
import os, gc, re, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print('Imports OK')

In [ ]:
# 0.2 — Paths Windows + cấu hình
BASE = r"d:\Datathon_Data"
CAT_FOCUS = 1050
WINDOW_START = pd.Timestamp('2025-11-09')
WINDOW_END = pd.Timestamp('2026-05-07')

DIM_DIR     = os.path.join(BASE, 'dim_listing')
SNAP_DIR    = os.path.join(BASE, 'fact_listing_snapshot')
CONTACT_DIR = os.path.join(BASE, 'fact_post_contact_interactions')
EVENTS_DIR  = os.path.join(BASE, 'fact_user_events')
TEST_USERS  = os.path.join(BASE, 'test', 'test_users.parquet')
OUT_DIR     = os.path.join(BASE, 'eda', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# POSITIVE event_type (theo README.md + Cursor.md)
POSITIVE_EVENTS = ('view_phone', 'contact_chat', 'other_interaction', 'contact_zalo', 'contact_sms')

print('BASE:', BASE)
print('Window:', WINDOW_START.date(), '→', WINDOW_END.date())
print('CAT_FOCUS:', CAT_FOCUS)

In [ ]:
# 0.3 — DuckDB connect (RAM-safe)
con = duckdb.connect(database=':memory:')
con.execute("PRAGMA memory_limit='8GB'")
con.execute("PRAGMA threads=4")
print('DuckDB:', con.execute('SELECT version()').fetchone()[0])

In [ ]:
# 0.4 — Helper: in báo cáo Data Quality đồng nhất
QUALITY_LOG = []

def qlog(group, issue, n_issue, total, action):
    pct = (n_issue / total * 100) if total else 0
    QUALITY_LOG.append({
        'group': group, 'issue': issue,
        'count': int(n_issue), 'total': int(total),
        'pct': round(pct, 3), 'action': action,
    })
    print(f'  [{group}] {issue:40s} | {n_issue:>8,} / {total:>8,} ({pct:5.2f}%) → {action}')

print('Helper qlog() ready.')

## Zone 1 — Nạp dữ liệu thô (raw)

Mục tiêu: chỉ load + filter `category=1050`, CHƯA xử lý gì cả. Tiền xử lý ở Zone 2-6.

### 1.1 — `dim_listing` → filter cat=1050

In [ ]:
# 1.1 — Load dim_listing rồi filter cat=1050
import glob
dim_files = sorted(glob.glob(os.path.join(DIM_DIR, '*.parquet')))
print(f'dim_listing files: {len(dim_files)}')

dim_dataset = ds.dataset(dim_files, format='parquet')
df_dim_raw = dim_dataset.to_table(filter=ds.field('category') == CAT_FOCUS).to_pandas()
print(f'df_dim_raw (cat=1050): {len(df_dim_raw):,} rows, {df_dim_raw.shape[1]} cols')
df_dim_raw.head(3)

### 1.2 — DuckDB VIEW `ev_raw_1050` cho `fact_user_events` (41GB — KHÔNG load pandas)

In [ ]:
# 1.2 — VIEW ev_raw_1050 (CHƯA xử lý)
con.execute(f"""
CREATE OR REPLACE VIEW ev_raw_1050 AS
SELECT *
FROM read_parquet('{EVENTS_DIR}/*.parquet', union_by_name=true)
WHERE category = {CAT_FOCUS}
""")
n_ev = con.execute('SELECT COUNT(*) FROM ev_raw_1050').fetchone()[0]
print(f'ev_raw_1050: {n_ev:,} rows (DuckDB view — không load pandas)')

### 1.3 — VIEW `contact_raw_1050` cho `fact_post_contact_interactions`

In [ ]:
# 1.3 — VIEW contact_raw_1050
con.execute(f"""
CREATE OR REPLACE VIEW contact_raw_1050 AS
SELECT *
FROM read_parquet('{CONTACT_DIR}/*.parquet', union_by_name=true)
WHERE category = {CAT_FOCUS}
""")
n_ct = con.execute('SELECT COUNT(*) FROM contact_raw_1050').fetchone()[0]
print(f'contact_raw_1050: {n_ct:,} rows')

### 1.4 — VIEW `snap_raw_1050` (INNER JOIN _item_ids vì snapshot không có cột category)

In [ ]:
# 1.4 — Tạo _item_ids từ df_dim_raw → JOIN vào snapshot
_item_ids = pd.DataFrame({'item_id': df_dim_raw['item_id'].unique()})
con.register('_item_ids_df', _item_ids)
con.execute('CREATE OR REPLACE TABLE _item_ids AS SELECT item_id FROM _item_ids_df')
print(f'_item_ids: {len(_item_ids):,} unique item_id')

con.execute(f"""
CREATE OR REPLACE VIEW snap_raw_1050 AS
SELECT s.*
FROM read_parquet('{SNAP_DIR}/*.parquet', union_by_name=true) s
INNER JOIN _item_ids i ON s.item_id = i.item_id
""")
n_snap = con.execute('SELECT COUNT(*) FROM snap_raw_1050').fetchone()[0]
print(f'snap_raw_1050: {n_snap:,} rows')

### 1.5 — `test_users.parquet`

In [ ]:
# 1.5 — test_users
df_test_users = pd.read_parquet(TEST_USERS)
print(f'test_users: {len(df_test_users):,} unique user_id, cols={list(df_test_users.columns)}')
df_test_users.head(3)

### 1.6 — Time coverage check

In [ ]:
# 1.6 — Time coverage min/max
print('--- Time coverage ---')
print('dim_listing.posted_date:',
      df_dim_raw['posted_date'].min(), '→', df_dim_raw['posted_date'].max())

ev_ts = con.execute('SELECT MIN(event_ts), MAX(event_ts) FROM ev_raw_1050').fetchone()
print('events.event_ts:        ', ev_ts[0], '→', ev_ts[1])

ct_dt = con.execute('SELECT MIN(date), MAX(date) FROM contact_raw_1050').fetchone()
print('contact.date:           ', ct_dt[0], '→', ct_dt[1])

snap_dt = con.execute('SELECT MIN(date), MAX(date) FROM snap_raw_1050').fetchone()
print('snapshot.date:          ', snap_dt[0], '→', snap_dt[1])

## Zone 2 — Tiền xử lý Nhóm A: Integrity & Cross-table

5 issues từ Cursor.md:
- **A.1** Duplicate `item_id` trong `dim_listing`
- **A.2** `contacts_24h > views_24h` (logically impossible)
- **A.3** `is_contact=1` nhưng `event_type='pageview'` (cờ sai)
- **A.4** `events.category != dim.category` (mismatch khi join)
- **A.5** Orphan: `item_id` trong events/contact không có trong dim_listing

### A.1 — Duplicate item_id trong dim_listing

In [ ]:
# A.1 — Dedupe item_id (giữ row có images_count cao nhất)
dup_mask = df_dim_raw['item_id'].duplicated(keep=False)
n_dup = dup_mask.sum()
total_dim = len(df_dim_raw)

df_dim = (df_dim_raw
          .sort_values('images_count', ascending=False, na_position='last')
          .drop_duplicates(subset='item_id', keep='first')
          .reset_index(drop=True))

n_drop = total_dim - len(df_dim)
qlog('A', 'A.1 Duplicate item_id', n_drop, total_dim,
     f'Dropped {n_drop}, kept row with max images_count')
print(f'df_dim sau dedupe: {len(df_dim):,} rows')

### A.2 — `contacts_24h > views_24h` trong snapshot

In [ ]:
# A.2 — Cap contacts_24h = MIN(contacts_24h, views_24h)
res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN contacts_24h > views_24h THEN 1 ELSE 0 END) AS n_bad
FROM snap_raw_1050
""").fetchone()
total_snap, n_bad_snap = res

con.execute("""
CREATE OR REPLACE VIEW snap_a2 AS
SELECT
  item_id, date, views_24h, listing_age_days,
  LEAST(contacts_24h, views_24h) AS contacts_24h_capped,
  contacts_24h AS contacts_24h_raw
FROM snap_raw_1050
""")
qlog('A', 'A.2 contacts > views', n_bad_snap, total_snap, 'Capped contacts = min(contacts, views)')

### A.3 — `is_contact` vs `event_type` consistency

In [ ]:
# A.3 — Trust event_type, recompute is_contact
pos_sql = "(" + ",".join(f"'{e}'" for e in POSITIVE_EVENTS) + ")"

res = con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE
    WHEN (event_type IN {pos_sql}) AND COALESCE(is_contact, 0) = 0 THEN 1
    WHEN (event_type NOT IN {pos_sql}) AND COALESCE(is_contact, 0) = 1 THEN 1
    ELSE 0
  END) AS n_bad
FROM ev_raw_1050
""").fetchone()
total_ev, n_bad_ev = res

con.execute(f"""
CREATE OR REPLACE VIEW ev_a3 AS
SELECT
  * EXCLUDE (is_contact),
  CASE WHEN event_type IN {pos_sql} THEN 1 ELSE 0 END AS is_contact
FROM ev_raw_1050
""")
qlog('A', 'A.3 is_contact vs event_type', n_bad_ev, total_ev, 'Recomputed from event_type')

### A.4 — `events.category` vs `dim.category` mismatch

In [ ]:
# A.4 — Check events có item_id trỏ về dim có category khác 1050
con.register('dim_df', df_dim[['item_id','category']])
res = con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN d.category IS NOT NULL AND d.category != {CAT_FOCUS} THEN 1 ELSE 0 END) AS n_bad
FROM ev_a3 e
LEFT JOIN dim_df d ON e.item_id = d.item_id
""").fetchone()
total_ev2, n_bad_cat = res
qlog('A', 'A.4 events.cat != dim.cat', n_bad_cat, total_ev2, 'Trust dim.category, drop mismatch khi join (A.5)')

### A.5 — Orphan items (events / contact có item_id không trong dim_listing)

In [ ]:
# A.5 — Drop orphan
con.execute('CREATE OR REPLACE TABLE _valid_items AS SELECT item_id FROM dim_df')

res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN v.item_id IS NULL THEN 1 ELSE 0 END) AS n_orphan
FROM ev_a3 e
LEFT JOIN _valid_items v ON e.item_id = v.item_id
""").fetchone()
total_ev3, n_orphan = res

con.execute("""
CREATE OR REPLACE VIEW ev_clean AS
SELECT e.*
FROM ev_a3 e
INNER JOIN _valid_items v ON e.item_id = v.item_id
""")
qlog('A', 'A.5 Orphan items in events', n_orphan, total_ev3, 'Dropped via INNER JOIN _valid_items')

res_ct = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN v.item_id IS NULL THEN 1 ELSE 0 END) AS n_orphan
FROM contact_raw_1050 c
LEFT JOIN _valid_items v ON c.item_id = v.item_id
""").fetchone()
total_ct, n_orphan_ct = res_ct

con.execute("""
CREATE OR REPLACE VIEW contact_clean AS
SELECT c.*
FROM contact_raw_1050 c
INNER JOIN _valid_items v ON c.item_id = v.item_id
""")
qlog('A', 'A.5 Orphan items in contact', n_orphan_ct, total_ct, 'Dropped via INNER JOIN _valid_items')

## Zone 3 — Tiền xử lý Nhóm B: Date/Time

4 issues:
- **B.1** Future leak (`posted_date` hoặc `event_ts` > WINDOW_END)
- **B.2** `expected_expired_date < posted_date` (invalid)
- **B.3** Timezone (`event_ts` UTC vs ICT)
- **B.4** `date` column vs `DATE(event_ts)` mismatch

### B.1 — Future leak

In [ ]:
# B.1 — Drop rows future
end_str = WINDOW_END.strftime('%Y-%m-%d')

# Dim listing: posted_date > end
n_future_post = (df_dim['posted_date'] > WINDOW_END.date()).sum()
total_dim2 = len(df_dim)
df_dim = df_dim[df_dim['posted_date'] <= WINDOW_END.date()].reset_index(drop=True)
qlog('B', 'B.1 posted_date future', n_future_post, total_dim2, f'Dropped {n_future_post}')

# Events: event_ts > end
res = con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN CAST(event_ts AS DATE) > DATE '{end_str}' THEN 1 ELSE 0 END) AS n_bad
FROM ev_clean
""").fetchone()
total_ev_b1, n_future_ev = res

con.execute(f"""
CREATE OR REPLACE VIEW ev_b1 AS
SELECT * FROM ev_clean WHERE CAST(event_ts AS DATE) <= DATE '{end_str}'
""")
qlog('B', 'B.1 event_ts future', n_future_ev, total_ev_b1, f'Dropped via WHERE')

### B.2 — `expected_expired_date < posted_date`

In [ ]:
# B.2 — Set expired = NULL khi invalid (không drop listing)
mask_b2 = (df_dim['expected_expired_date'].notna() &
           df_dim['posted_date'].notna() &
           (df_dim['expected_expired_date'] < df_dim['posted_date']))
n_b2 = mask_b2.sum()
df_dim.loc[mask_b2, 'expected_expired_date'] = pd.NaT
qlog('B', 'B.2 expired < posted', n_b2, len(df_dim), 'Set expected_expired_date=NULL')

### B.3 — Timezone của `event_ts`

In [ ]:
# B.3 — Phát hiện timezone
sample = con.execute("SELECT event_ts FROM ev_b1 LIMIT 5").fetchall()
print('Sample event_ts:', sample)

# Check timezone info từ schema (DuckDB)
schema_info = con.execute("DESCRIBE ev_b1").fetchdf()
ts_dtype = schema_info[schema_info['column_name']=='event_ts']['column_type'].iloc[0]
print('event_ts dtype:', ts_dtype)

# Nếu là TIMESTAMP (naive) thì convert sang ICT
con.execute("""
CREATE OR REPLACE VIEW ev_b3 AS
SELECT
  * EXCLUDE (event_ts),
  CAST(event_ts AS TIMESTAMP) AS event_ts
FROM ev_b1
""")
qlog('B', 'B.3 Timezone normalize', 0, 0, f'Treated as ICT naive (dtype={ts_dtype})')

### B.4 — `date` vs `DATE(event_ts)` mismatch

In [ ]:
# B.4 — Recompute date từ event_ts
res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN CAST(event_ts AS DATE) != date THEN 1 ELSE 0 END) AS n_bad
FROM ev_b3
""").fetchone()
total_ev_b4, n_bad_b4 = res

con.execute("""
CREATE OR REPLACE VIEW ev_b4 AS
SELECT
  * EXCLUDE (date),
  CAST(event_ts AS DATE) AS date
FROM ev_b3
""")
qlog('B', 'B.4 date != DATE(event_ts)', n_bad_b4, total_ev_b4, 'Recomputed date from event_ts')

## Zone 4 — Tiền xử lý Nhóm C: Numeric outliers

5 issues:
- **C.1** `dwell_time_sec` median check → chia /1000 nếu cần (rule Cursor.md: đang ở ms)
- **C.2** `views_24h = 0` (flag, không drop)
- **C.3** `area_sqm` outlier (phòng trọ kỳ vọng 10–60 m²)
- **C.4** bedrooms/bathrooms/floors/width_m null rate (cat 1050 thường null nhiều)
- **C.5** `position` dtype

### C.1 — `dwell_time_sec`: chia /1000 nếu đang ở ms

In [ ]:
# C.1 — Median check + scale
stats = con.execute("""
SELECT
  COUNT(*) AS total,
  MEDIAN(dwell_time_sec) AS med,
  QUANTILE_CONT(dwell_time_sec, 0.99) AS p99,
  MAX(dwell_time_sec) AS mx
FROM ev_b4
WHERE dwell_time_sec IS NOT NULL
""").fetchone()
total_d, med_d, p99_d, max_d = stats
print(f'dwell_time_sec — median={med_d:.1f}, P99={p99_d:.1f}, max={max_d:.1f}')

scale_factor = 1000.0 if med_d and med_d > 1000 else 1.0
cap_value = p99_d / scale_factor

con.execute(f"""
CREATE OR REPLACE VIEW ev_c1 AS
SELECT
  * EXCLUDE (dwell_time_sec),
  LEAST(dwell_time_sec / {scale_factor}, {cap_value}) AS dwell_time_sec
FROM ev_b4
""")
qlog('C', 'C.1 dwell_time scale', total_d, total_d,
     f'/{scale_factor:.0f} + cap P99={cap_value:.1f}s')

### C.2 — `views_24h = 0` flag

In [ ]:
# C.2 — Flag is_zero_view (không drop)
res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN views_24h = 0 OR views_24h IS NULL THEN 1 ELSE 0 END) AS n_zero
FROM snap_a2
""").fetchone()
total_snap_c2, n_zero = res

con.execute("""
CREATE OR REPLACE VIEW snap_clean AS
SELECT
  *,
  CASE WHEN views_24h = 0 OR views_24h IS NULL THEN 1 ELSE 0 END AS is_zero_view
FROM snap_a2
""")
qlog('C', 'C.2 views_24h = 0', n_zero, total_snap_c2, 'Flagged is_zero_view, kept')

### C.3 — `area_sqm` outlier (phòng trọ kỳ vọng 10–60 m²)

In [ ]:
# C.3 — Set NULL nếu area_sqm < 5 hoặc > 200
mask_c3 = df_dim['area_sqm'].notna() & ((df_dim['area_sqm'] < 5) | (df_dim['area_sqm'] > 200))
n_c3 = int(mask_c3.sum())
df_dim.loc[mask_c3, 'area_sqm'] = np.nan
qlog('C', 'C.3 area_sqm outlier', n_c3, len(df_dim), 'Set NULL when <5 or >200 m²')

# In thêm distribution
print('area_sqm describe (sau cleanup):')
print(df_dim['area_sqm'].describe().round(2))

### C.4 — Null rate của bedrooms/bathrooms/floors/width_m

In [ ]:
# C.4 — Null rate per col, drop nếu null > 95%
NUMERIC_OPTIONAL = ['bedrooms', 'bathrooms', 'floors', 'width_m']
drop_cols = []
for col in NUMERIC_OPTIONAL:
    if col in df_dim.columns:
        null_rate = df_dim[col].isna().mean()
        action = 'Dropped (null>95%)' if null_rate > 0.95 else f'Kept (null={null_rate*100:.1f}%)'
        qlog('C', f'C.4 {col} null rate', int(df_dim[col].isna().sum()), len(df_dim), action)
        if null_rate > 0.95:
            drop_cols.append(col)
df_dim = df_dim.drop(columns=drop_cols, errors='ignore')
print(f'Dropped: {drop_cols}')
print(f'df_dim cols còn lại: {list(df_dim.columns)}')

### C.5 — `position` dtype

In [ ]:
# C.5 — Ép position về int (hiện đang float)
res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN position IS NOT NULL THEN 1 ELSE 0 END) AS n_notnull
FROM ev_c1
""").fetchone()
total_pos, n_notnull_pos = res

con.execute("""
CREATE OR REPLACE VIEW ev_c5 AS
SELECT
  * EXCLUDE (position),
  TRY_CAST(position AS INTEGER) AS position
FROM ev_c1
""")
qlog('C', 'C.5 position → int', n_notnull_pos, total_pos, 'Cast float → INTEGER')

## Zone 5 — Tiền xử lý Nhóm D: Categorical

4 issues + 1 bonus:
- **D.1** Parse `price_bucket` ("Dưới X", "Trên Y", "Thỏa thuận") → `price_low_vnd`, `price_high_vnd`, `price_mid_vnd`, `is_negotiable`
- **D.1.bonus** Tạo `price_per_sqm = price_mid_vnd / area_sqm` (Value-for-money)
- **D.2** Category-adtype mismatch (1050 phòng trọ thì `ad_type` phải là `let`)
- **D.3** String whitespace trim + normalize
- **D.4** `purchased` ≠ ground truth (warning, không drop)

### D.1 — Parse `price_bucket` thành numeric

In [ ]:
# D.1 — Parser price_bucket
# Format thực tế cat 1050: "3M–5M/tháng", "<2M/tháng", ">30M/tháng", "2B–3B", "<500M"
# M = triệu (1e6), B = tỷ (1e9), K = nghìn (1e3)
UNIT_MULT = {'M': 1e6, 'B': 1e9, 'K': 1e3}

def _to_vnd(num_str, unit):
    try:
        n = float(num_str.replace(',', '.'))
    except Exception:
        return None
    return n * UNIT_MULT.get(unit.upper(), 1.0)

def parse_price_bucket(s):
    """Trả về (low_vnd, high_vnd, is_negotiable). None nếu không parse được."""
    if not isinstance(s, str) or not s.strip():
        return (None, None, None)
    txt = s.strip()
    txt_lower = txt.lower()
    if 'thỏa thuận' in txt_lower or 'thoa thuan' in txt_lower:
        return (None, None, True)

    # Pattern range: "3M–5M/tháng" hoặc "3M-5M" hoặc "3M~5M" (dùng en-dash hoặc hyphen)
    m = re.search(r'([\d.,]+)\s*([MBK])\s*[-–~]\s*([\d.,]+)\s*([MBK])', txt)
    if m:
        return (_to_vnd(m.group(1), m.group(2)), _to_vnd(m.group(3), m.group(4)), False)

    # Pattern "<X đơn vị" hoặc "Dưới X"
    m = re.search(r'(?:^<|dưới|duoi)\s*([\d.,]+)\s*([MBK])', txt)
    if m:
        high = _to_vnd(m.group(1), m.group(2))
        return (0.0, high, False)

    # Pattern ">X đơn vị" hoặc "Trên X"
    m = re.search(r'(?:^>|trên|tren)\s*([\d.,]+)\s*([MBK])', txt)
    if m:
        low = _to_vnd(m.group(1), m.group(2))
        return (low, low * 1.5 if low else None, False)

    # Fallback pattern cũ: "X triệu - Y triệu"
    m = re.search(r'([\d.,]+)\s*(tỷ|triệu|nghìn)\s*(?:/tháng)?\s*[-–~]\s*([\d.,]+)\s*(tỷ|triệu|nghìn)', txt_lower)
    if m:
        unit_map = {'tỷ': 'B', 'triệu': 'M', 'nghìn': 'K'}
        return (_to_vnd(m.group(1), unit_map[m.group(2)]),
                _to_vnd(m.group(3), unit_map[m.group(4)]), False)

    return (None, None, None)

parsed = df_dim['price_bucket'].apply(parse_price_bucket)
df_dim['price_low_vnd']  = [p[0] for p in parsed]
df_dim['price_high_vnd'] = [p[1] for p in parsed]
df_dim['is_negotiable']  = [p[2] for p in parsed]
df_dim['price_mid_vnd']  = (df_dim['price_low_vnd'].fillna(0) + df_dim['price_high_vnd'].fillna(0)) / 2
df_dim.loc[(df_dim['price_low_vnd'].isna()) & (df_dim['price_high_vnd'].isna()), 'price_mid_vnd'] = np.nan

n_parsed = df_dim['price_mid_vnd'].notna().sum()
n_negotiable = (df_dim['is_negotiable'] == True).sum()
n_unparsed = df_dim['price_bucket'].notna().sum() - n_parsed - n_negotiable
qlog('D', 'D.1 price_bucket parsed', n_parsed, len(df_dim), 'Created price_low/high/mid_vnd')
qlog('D', 'D.1 price negotiable', n_negotiable, len(df_dim), 'Flagged is_negotiable=True')
qlog('D', 'D.1 price unparsed', n_unparsed, len(df_dim), 'NULL (giữ lại để debug)')

print('Distribution price_mid_vnd (triệu VNĐ):')
print((df_dim['price_mid_vnd'] / 1e6).describe().round(2))

### D.1.bonus — `price_per_sqm` (Value-for-money)

In [ ]:
# D.1.bonus — price_per_sqm = price_mid_vnd / area_sqm
mask_ok = df_dim['price_mid_vnd'].notna() & df_dim['area_sqm'].notna() & (df_dim['area_sqm'] > 0)
df_dim['price_per_sqm'] = np.where(
    mask_ok,
    df_dim['price_mid_vnd'] / df_dim['area_sqm'],
    np.nan,
)
n_pps = int(mask_ok.sum())

# Cap outlier ở P99 (giá/m² cực cao có thể là tin sai)
if n_pps > 0:
    p99 = df_dim.loc[mask_ok, 'price_per_sqm'].quantile(0.99)
    p01 = df_dim.loc[mask_ok, 'price_per_sqm'].quantile(0.01)
    df_dim['price_per_sqm'] = df_dim['price_per_sqm'].clip(lower=p01, upper=p99)
    print(f'price_per_sqm — cap P01={p01/1e3:.1f}K, P99={p99/1e3:.1f}K (đồng/m²)')

qlog('D', 'D.1.bonus price_per_sqm', n_pps, len(df_dim), 'Tạo + cap P01-P99 outlier')

print('\nDistribution price_per_sqm (nghìn đồng/m²):')
print((df_dim['price_per_sqm'] / 1e3).describe().round(1))

### D.2 — Category–adtype mismatch

In [ ]:
# D.2 — Cat 1050 (phòng trọ) thì ad_type phải là 'let'
vc_ad = df_dim['ad_type'].value_counts(dropna=False)
print('ad_type distribution:')
print(vc_ad)

n_sell = (df_dim['ad_type'] == 'sell').sum()
qlog('D', 'D.2 ad_type=sell ở cat 1050', n_sell, len(df_dim),
     'Flag suspicious (phòng trọ phải let)')

### D.3 — String whitespace trim + lower

In [ ]:
# D.3 — Normalize text cols
TEXT_COLS = ['furnishing', 'house_type', 'direction', 'district_name', 'ward_name', 'city_name', 'seller_type']
n_normalized = 0
for col in TEXT_COLS:
    if col in df_dim.columns:
        before = df_dim[col].astype(str).copy()
        df_dim[col] = df_dim[col].astype(str).str.strip()
        # Chỉ lower cho cột phân loại (furnishing, house_type, seller_type)
        if col in ('furnishing', 'house_type', 'seller_type', 'direction'):
            df_dim[col] = df_dim[col].str.lower()
        n_diff = (before != df_dim[col]).sum()
        n_normalized += n_diff
qlog('D', 'D.3 text normalize', n_normalized, len(df_dim) * len(TEXT_COLS), 'Trim + lower')

# Restore NaN cho 'nan' string
for col in TEXT_COLS:
    if col in df_dim.columns:
        df_dim.loc[df_dim[col].isin(['nan', 'none', '']), col] = np.nan

### D.4 — `purchased` không phải ground truth

In [ ]:
# D.4 — Warning rõ ràng
n_purchased = con.execute("SELECT SUM(CAST(purchased AS INTEGER)) FROM contact_clean").fetchone()[0]
total_ct = con.execute("SELECT COUNT(*) FROM contact_clean").fetchone()[0]
qlog('D', 'D.4 purchased flag', int(n_purchased or 0), total_ct,
     'WARNING: purchased là prediction nội bộ Chợ Tốt, KHÔNG phải ground truth')
print('\n⚠️  CẢNH BÁO: cột `purchased` là dự đoán nội bộ của Chợ Tốt — KHÔNG dùng làm label thật!')

## Zone 6 — Tiền xử lý Nhóm E: User behavior

2 issues:
- **E.1** Non-login `user_id` thay đổi mỗi session → tạo `user_scope_id = session_id` cho non-login
- **E.2** Session is_login mixing (1 session có cả login + non-login) → first-state wins

**Lưu ý**: Theo confirm với user, scope phân tích = **login + non-login session-scoped** (giữ cả 2).

### E.1 — Tạo `user_scope_id` cho non-login

In [ ]:
# E.1 — Non-login: user_scope_id = session_id; login: user_scope_id = user_id
res = con.execute("""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_login = 'non-login' THEN 1 ELSE 0 END) AS n_nonlogin
FROM ev_c5
""").fetchone()
total_e1, n_nonlogin = res

con.execute("""
CREATE OR REPLACE VIEW ev_e1 AS
SELECT
  *,
  CASE WHEN is_login = 'login' THEN user_id ELSE session_id END AS user_scope_id
FROM ev_c5
""")
qlog('E', 'E.1 non-login → user_scope_id', n_nonlogin, total_e1,
     'Tạo user_scope_id: login=user_id, non-login=session_id')

### E.2 — Session is_login mixing

In [ ]:
# E.2 — Đếm session có cả login + non-login
res = con.execute("""
SELECT
  COUNT(*) AS total_sessions,
  SUM(CASE WHEN n_states > 1 THEN 1 ELSE 0 END) AS n_mixed
FROM (
  SELECT session_id, COUNT(DISTINCT is_login) AS n_states
  FROM ev_e1
  GROUP BY session_id
) t
""").fetchone()
total_sess, n_mixed = res

# Action: first-state per session (theo MIN event_ts). Login WINS nếu có.
con.execute("""
CREATE OR REPLACE VIEW _session_state AS
SELECT
  session_id,
  MAX(CASE WHEN is_login = 'login' THEN 1 ELSE 0 END) AS session_has_login
FROM ev_e1
GROUP BY session_id
""")

con.execute("""
CREATE OR REPLACE VIEW ev_e2 AS
SELECT
  e.* EXCLUDE (is_login),
  CASE WHEN s.session_has_login = 1 THEN 'login' ELSE 'non-login' END AS is_login
FROM ev_e1 e
LEFT JOIN _session_state s ON e.session_id = s.session_id
""")

qlog('E', 'E.2 session is_login mixed', n_mixed, total_sess, 'Login wins per session')

# Tạo VIEW cuối cùng: ev_final
con.execute("CREATE OR REPLACE VIEW ev_final AS SELECT * FROM ev_e2")
print('VIEW ev_final ready — đây là events đã xử lý xong Group A-E.')

## Zone 7 — Tổng hợp Data Quality Report

Bảng tổng hợp tất cả các issue đã xử lý (Group A-E). Export ra CSV để các phase sau (modeling) tham chiếu.

In [ ]:
# 7.1 — Build DataFrame từ QUALITY_LOG
df_quality = pd.DataFrame(QUALITY_LOG)
print(f'Tổng số issue đã xử lý: {len(df_quality)}')
print('\nGroup-by summary:')
print(df_quality.groupby('group').agg(n_issues=('issue','count'),
                                      total_count=('count','sum')).reset_index())

In [ ]:
# 7.2 — Hiển thị full report
pd.set_option('display.max_colwidth', 80)
df_quality_display = df_quality[['group','issue','count','pct','action']].copy()
df_quality_display.columns = ['Nhóm','Vấn đề','Số rows','% tổng','Hành động']
df_quality_display

In [ ]:
# 7.3 — Export CSV
out_csv = os.path.join(OUT_DIR, 'cat1050_v3_quality_report.csv')
df_quality.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'✅ Saved: {out_csv}')
print(f'   {len(df_quality)} issues × 6 cols')

### Checkpoint sau Zone 7

Sau Zone 7, các object dùng được trong EDA:
- `df_dim` — pandas DataFrame dim_listing đã clean (~9K rows)
- `ev_final` — DuckDB VIEW events đã clean (Group A-E)
- `contact_clean` — DuckDB VIEW contact đã clean
- `snap_clean` — DuckDB VIEW snapshot đã clean (có flag is_zero_view)
- `df_test_users` — danh sách user_id test

**Cleanup RAM**: drop df_dim_raw + collect GC.

In [ ]:
# 7.4 — Cleanup RAM
del df_dim_raw
gc.collect()
print('df_dim_raw dropped. df_dim còn lại:', df_dim.shape)
print('df_dim columns:', list(df_dim.columns))

## Zone 8 — EDA Supply (Cung): Phân tích tin đăng

Mỗi chart phía dưới đều có 3 phần giải thích:
- **Mô tả biểu đồ**: chart thể hiện gì.
- **Insight nghiệp vụ**: chỉ ra điểm nghẽn / outlier + so sánh với mặt bằng chung.
- **Hướng giải pháp**: Quick-win (vá lỗi) + Deep-dive (root cause).

### Chart S1 — Phân phối giá phòng trọ (`price_mid_vnd`)

In [ ]:
# S1 — Histogram price_mid_vnd
s1 = df_dim['price_mid_vnd'].dropna() / 1e6  # đơn vị triệu VNĐ
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(s1, bins=40, color='steelblue', edgecolor='white')
ax.axvline(s1.median(), color='red', linestyle='--', label=f'Trung vị = {s1.median():.1f} triệu')
ax.axvline(s1.mean(), color='orange', linestyle='--', label=f'Trung bình = {s1.mean():.1f} triệu')
ax.set_title('Phân phối Giá Phòng Trọ (cat 1050)')
ax.set_xlabel('Giá trung bình (triệu VNĐ/tháng)')
ax.set_ylabel('Số tin đăng')
ax.legend()
plt.tight_layout(); plt.show()
print('Stats:', s1.describe().round(2).to_dict())

**Mô tả biểu đồ:** Phân phối giá trung bình các tin đăng phòng trọ. Đường đỏ = trung vị, đường cam = trung bình.

**Insight nghiệp vụ:** Phân phối lệch phải (skewed right) — đa số tin tập trung ở phân khúc thấp (1–5 triệu/tháng), một số tin "luxury" kéo dài đuôi phải. Nếu trung bình lệch xa trung vị → outlier giá cao có thể là nhà nguyên căn bị mis-categorize vào cat 1050.

**Hướng giải pháp:**
- *Quick-win*: thêm bộ lọc validate giá khi đăng tin — phòng trọ thường < 10 triệu/tháng, tin > 15 triệu cần kiểm duyệt.
- *Deep-dive*: nghiên cứu sự khác biệt cung-cầu theo phân khúc giá để xem có bias đăng tin "fake luxury" không.

### Chart S2 — Phân phối diện tích (`area_sqm`)

In [ ]:
# S2 — Histogram area_sqm
s2 = df_dim['area_sqm'].dropna()
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(s2, bins=30, color='seagreen', edgecolor='white')
ax.axvspan(10, 30, alpha=0.15, color='red', label='Kỳ vọng phòng trọ: 10–30 m²')
ax.axvline(s2.median(), color='red', linestyle='--', label=f'Trung vị = {s2.median():.1f} m²')
ax.set_title('Phân phối Diện tích Phòng Trọ')
ax.set_xlabel('Diện tích (m²)')
ax.set_ylabel('Số tin đăng')
ax.legend()
plt.tight_layout(); plt.show()

pct_in_range = ((s2 >= 10) & (s2 <= 30)).mean() * 100
pct_above = (s2 > 30).mean() * 100
print(f'Trong khoảng 10–30 m²: {pct_in_range:.1f}%')
print(f'Trên 30 m² (nghi vấn nhà nguyên căn): {pct_above:.1f}%')

**Mô tả biểu đồ:** Phân phối diện tích phòng trọ. Vùng đỏ nhạt = khoảng kỳ vọng (10–30 m²).

**Insight nghiệp vụ:** Phòng trọ thực sự nên < 30 m². Tỷ lệ tin > 30 m² cao bất thường → có thể là **nhà nguyên căn / căn hộ mini** bị đăng nhầm category. Đây là điểm nghẽn data quality ảnh hưởng đến recommender vì user tìm "phòng trọ rẻ" sẽ ngỡ ngàng khi click vào nhà nguyên căn 50 m² giá cao.

**Hướng giải pháp:**
- *Quick-win*: hard validation khi đăng — diện tích > 40 m² thì bắt buộc chọn cat 1020/1030.
- *Deep-dive*: hành vi seller cố tình đăng sai cat để "leo top" search phòng trọ (segment có demand cao hơn nhà nguyên căn).

### Chart S3 — Phân phối nội thất (`furnishing`)

In [ ]:
# S3 — Bar chart furnishing
s3 = df_dim['furnishing'].fillna('(không khai)').value_counts()
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(s3)), s3.values, color=sns.color_palette('viridis', len(s3)))
ax.set_xticks(range(len(s3)))
ax.set_xticklabels(s3.index, rotation=30, ha='right')
ax.set_title('Phân phối Tình trạng Nội thất')
ax.set_ylabel('Số tin đăng')
for i, v in enumerate(s3.values):
    ax.text(i, v + max(s3.values)*0.01, f'{v:,}\n({v/len(df_dim)*100:.1f}%)',
            ha='center', fontsize=9)
plt.tight_layout(); plt.show()
print(s3)

**Mô tả biểu đồ:** Số lượng tin đăng theo từng loại nội thất.

**Insight nghiệp vụ:** Tỷ lệ "Nội thất đầy đủ" cho biết phân khúc trọ trung-cao cấp. Nếu tỷ lệ "(không khai)" cao → seller làm tin sơ sài, ảnh hưởng đến chất lượng search và conversion (xem Chart E2).

**Hướng giải pháp:**
- *Quick-win*: bắt buộc chọn `furnishing` khi đăng tin (UX required field).
- *Deep-dive*: kiểm tra xem tin "không khai nội thất" có CVR thấp hơn bao nhiêu để định lượng impact, sau đó push reminder cho seller bổ sung.

### Chart S4 — Top 10 quận theo số tin đăng (HCM vs HN)

In [ ]:
# S4 — Top 10 district by supply count
s4 = df_dim.groupby(['city_name','district_name']).size().reset_index(name='n').sort_values('n', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(12, 6))
labels = [f"{r['district_name']} ({r['city_name']})" for _, r in s4.iterrows()]
ax.barh(labels[::-1], s4['n'].values[::-1], color='coral')
ax.set_title('Top 15 Quận theo Số Tin Phòng Trọ')
ax.set_xlabel('Số tin đăng')
plt.tight_layout(); plt.show()

city_share = df_dim['city_name'].value_counts(normalize=True).head(5) * 100
print('Top 5 city share (%):'); print(city_share.round(2))

**Mô tả biểu đồ:** 15 quận có nhiều tin phòng trọ nhất, gắn kèm thành phố.

**Insight nghiệp vụ:** Cung phòng trọ tập trung mạnh ở HCM (thường > 70%) và HN. Các quận top thường là khu sinh viên / công nhân / gần KCN. Nếu cung tập trung quá ở vài quận thì user ở quận khác sẽ thiếu lựa chọn → recommender phải mở rộng radius khi tìm.

**Hướng giải pháp:**
- *Quick-win*: ở quận có cung thấp, recommender mở rộng sang quận lân cận (geo-fallback).
- *Deep-dive*: phân tích supply-demand gap (Zone 10) để xác định vùng "khát cung" cần marketing seller.

### Chart S5 — Agent vs Private + ảnh đăng kèm

In [ ]:
# S5 — seller_type và avg images_count
s5 = df_dim.groupby('seller_type').agg(n=('item_id','count'), avg_img=('images_count','mean')).reset_index()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.bar(s5['seller_type'].fillna('(NA)'), s5['n'], color=['#1f77b4','#ff7f0e','#888'])
ax1.set_title('Số tin theo Loại Người Đăng')
ax1.set_ylabel('Số tin')
for i, v in enumerate(s5['n'].values):
    ax1.text(i, v, f'{v:,}', ha='center', va='bottom')

ax2.bar(s5['seller_type'].fillna('(NA)'), s5['avg_img'], color=['#1f77b4','#ff7f0e','#888'])
ax2.set_title('Trung bình Số ảnh / tin theo Loại')
ax2.set_ylabel('avg images_count')
for i, v in enumerate(s5['avg_img'].values):
    ax2.text(i, v, f'{v:.1f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
print(s5.round(2))

**Mô tả biểu đồ:** Trái — số lượng tin theo loại seller (agent / private). Phải — trung bình số ảnh upload.

**Insight nghiệp vụ:** Trong phòng trọ, **private (cá nhân = chủ trọ)** thường chiếm đa số. Nếu agent upload nhiều ảnh hơn → tin agent có thể được ưu tiên hiển thị nhờ visual richer, nhưng người thuê thường thích chủ trọ trực tiếp để tránh phí môi giới (xem Chart E1).

**Hướng giải pháp:**
- *Quick-win*: thêm badge "Chủ trọ" (private) vs "Môi giới" (agent) trên thẻ tin để user lọc nhanh.
- *Deep-dive*: phân tích CVR theo seller_type để xem user có thực sự thích private không (Zone 11 — Chart E1).

### Chart S6 — Số ảnh đính kèm theo bin

In [ ]:
# S6 — images_count bins
img = df_dim['images_count'].fillna(0).clip(0, 30)
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(img, bins=30, color='goldenrod', edgecolor='white')
ax.axvline(img.median(), color='red', linestyle='--', label=f'Trung vị = {img.median():.0f}')
ax.set_title('Phân phối Số Ảnh / tin (clip ở 30)')
ax.set_xlabel('Số ảnh')
ax.set_ylabel('Số tin')
ax.legend()
plt.tight_layout(); plt.show()

pct_low_img = (df_dim['images_count'].fillna(0) <= 1).mean() * 100
print(f'Tỷ lệ tin có ≤ 1 ảnh: {pct_low_img:.1f}% — nghi vấn spam / sơ sài')

**Mô tả biểu đồ:** Phân phối số ảnh trên mỗi tin (cap ở 30 để dễ đọc).

**Insight nghiệp vụ:** Tin có 0–1 ảnh là "tin sơ sài" — gần như chắc chắn CVR thấp. Nếu tỷ lệ này > 15% thì có một lượng đáng kể "tin rác" làm loãng feed.

**Hướng giải pháp:**
- *Quick-win*: yêu cầu tối thiểu 3 ảnh khi đăng tin (UX validation). Tin 0 ảnh chuyển thành "tin nháp" cho đến khi seller bổ sung.
- *Deep-dive*: kiểm tra xem các tin 0–1 ảnh có thuộc về seller mới (cần onboarding) hay seller cũ (cần nhắc nhở), từ đó push notification đúng đối tượng.

### Chart S7 — Phân phối `price_per_sqm` (Value-for-money)

In [ ]:
# S7 — price_per_sqm distribution
s7 = (df_dim['price_per_sqm'].dropna() / 1e3)  # nghìn đồng/m²
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(s7, bins=40, color='mediumpurple', edgecolor='white')
ax.axvline(s7.median(), color='red', linestyle='--', label=f'Trung vị = {s7.median():.1f}K/m²')
ax.set_title('Phân phối Giá / m² (price_per_sqm)')
ax.set_xlabel('Nghìn đồng / m² / tháng')
ax.set_ylabel('Số tin')
ax.legend()
plt.tight_layout(); plt.show()

q1, q3 = s7.quantile([0.25, 0.75])
print(f'IQR price_per_sqm: {q1:.0f}K – {q3:.0f}K (đồng/m²)')
n_very_low = (s7 < s7.quantile(0.05)).sum()
print(f'Số tin có giá/m² cực thấp (< P5): {n_very_low:,} — nghi vấn tin ảo / sai diện tích')

**Mô tả biểu đồ:** Phân phối "giá thuê trên mỗi mét vuông" — metric quan trọng đo "độ hời" của tin.

**Insight nghiệp vụ:** Phân phối lệch phải. Tin có `price_per_sqm` cực thấp (< P5) thường là **tin ảo** hoặc seller khai sai diện tích để "câu click". Tin có `price_per_sqm` cực cao (> P95) tập trung ở trung tâm Q1 / Q3 HCM — đắt nhưng vẫn có demand. Đây là feature quan trọng cho recommender (matching người thuê tìm "rẻ" với tin có price_per_sqm thấp ở quận họ chọn).

**Hướng giải pháp:**
- *Quick-win*: thêm chỉ số "đ/m²" hiển thị bên cạnh giá tổng để user so sánh nhanh.
- *Deep-dive*: nghiên cứu ngưỡng `price_per_sqm` hợp lý từng quận (Chart E7) để cảnh báo tin "giá quá hời = nghi tin ảo".

## Zone 9 — EDA Demand (Cầu): Phân tích user events & contacts

**Nguyên tắc RAM**: aggregate trong DuckDB SQL trước, chỉ `.df()` kết quả nhỏ rồi vẽ chart.

### Chart D1 — Funnel hành vi (theo event_type, per-item)

In [ ]:
# D1 — Funnel per-item: % tin đăng nhận được mỗi loại tương tác
# Lưu ý: surface thực tế chỉ có ad_view/adview → không dùng làm bước. Thay vào đó dùng event_type.
funnel_sql = """
WITH per_item AS (
  SELECT
    item_id,
    MAX(CASE WHEN event_type = 'pageview' THEN 1 ELSE 0 END) AS had_pageview,
    MAX(CASE WHEN event_type = 'other_interaction' THEN 1 ELSE 0 END) AS had_engage,
    MAX(CASE WHEN event_type IN ('view_phone','contact_chat') THEN 1 ELSE 0 END) AS had_intent,
    MAX(CASE WHEN event_type IN ('contact_zalo','contact_sms') THEN 1 ELSE 0 END) AS had_conv
  FROM ev_final
  GROUP BY item_id
)
SELECT
  SUM(had_pageview) AS step1_pageview,
  SUM(had_engage)   AS step2_engage,
  SUM(had_intent)   AS step3_intent,
  SUM(had_conv)     AS step4_conv,
  COUNT(*) AS n_items
FROM per_item
"""
df_funnel = con.execute(funnel_sql).df()
print(df_funnel.T.rename(columns={0:'count'}))

steps = ['1. Pageview\n(xem tin)',
         '2. Engagement\n(other_interaction)',
         '3. Intent\n(view_phone / chat)',
         '4. Conversion\n(zalo / sms)']
values = [int(df_funnel['step1_pageview'].iloc[0]),
          int(df_funnel['step2_engage'].iloc[0]),
          int(df_funnel['step3_intent'].iloc[0]),
          int(df_funnel['step4_conv'].iloc[0])]
total_items = int(df_funnel['n_items'].iloc[0])

fig, ax = plt.subplots(figsize=(13, 6))
colors = ['#4c72b0','#55a868','#c44e52','#8172b2']
bars = ax.bar(steps, values, color=colors)
ax.set_title(f'Funnel Per-Item — Cat 1050 ({total_items:,} tin có ≥ 1 event)')
ax.set_ylabel('Số tin có ít nhất 1 event của bước')

# Hiển thị số + % bên trên mỗi cột
for i, v in enumerate(values):
    pct = v / total_items * 100 if total_items else 0
    ax.text(i, v, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

# Drop rate giữa các bước (text giữa cột)
for i in range(1, len(values)):
    if values[i-1] > 0:
        drop_pct = (1 - values[i] / values[i-1]) * 100
        y_pos = (values[i-1] + values[i]) / 2
        ax.annotate(f'↓ {drop_pct:.1f}%', xy=(i - 0.5, y_pos),
                    ha='center', fontsize=11, color='red', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red'))

plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Funnel **per-item** — mỗi cột là **% tin đăng** có ít nhất 1 event của bước đó (không phải tổng events). Mũi tên đỏ = % drop giữa các bước.

**Lưu ý kỹ thuật:** Surface trong data chỉ có giá trị `ad_view`/`adview` (mọi event đều xem ad), nên funnel xây dựng theo `event_type` thay vì `surface`.

**Insight nghiệp vụ:** Điểm nghẽn lớn nhất thường ở **Bước 3 → Bước 4 (Intent → Conversion)** — user đã bấm xem SĐT hoặc mở chat nhưng không gọi/nhắn thực sự. Nguyên nhân thường gặp: số quá nhiều thông tin so sánh, seller không phản hồi (Chart E3), hoặc user dùng SĐT để gọi trực tiếp ngoài platform.

**Hướng giải pháp:**
- *Quick-win*: thêm tracking "đã gọi điện" sau khi click view_phone (web-to-call) để đo conversion thực.
- *Deep-dive*: phân tích cohort user click view_phone mà không convert — họ có quay lại sau N ngày không, có chuyển sang tin khác không.

### Chart D2 — CVR theo bins giá `price_mid_vnd`

In [ ]:
# D2 — CVR theo bins giá
con.register('dim_price', df_dim[['item_id','price_mid_vnd']])
sql = """
SELECT
  CASE
    WHEN d.price_mid_vnd IS NULL THEN '(NA)'
    WHEN d.price_mid_vnd < 2e6  THEN '< 2tr'
    WHEN d.price_mid_vnd < 4e6  THEN '2-4tr'
    WHEN d.price_mid_vnd < 6e6  THEN '4-6tr'
    WHEN d.price_mid_vnd < 8e6  THEN '6-8tr'
    WHEN d.price_mid_vnd < 12e6 THEN '8-12tr'
    ELSE '> 12tr'
  END AS price_bin,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
LEFT JOIN dim_price d ON e.item_id = d.item_id
GROUP BY 1
ORDER BY MIN(d.price_mid_vnd) NULLS LAST
"""
df_d2 = con.execute(sql).df()
print(df_d2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_d2['price_bin'], df_d2['cvr_pct'], color='teal')
ax.set_title('CVR (%) theo Phân khúc Giá')
ax.set_xlabel('Khoảng giá (triệu/tháng)')
ax.set_ylabel('CVR (%)')
overall_cvr = df_d2['n_contact'].sum() / df_d2['n_events'].sum() * 100
ax.axhline(overall_cvr, color='red', linestyle='--', label=f'CVR tổng = {overall_cvr:.2f}%')
for i, v in enumerate(df_d2['cvr_pct']):
    ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
ax.legend()
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Conversion Rate (CVR = contact / events) theo từng phân khúc giá. Đường đỏ = CVR trung bình toàn cat 1050.

**Insight nghiệp vụ:** Segment nào CVR cao > đường đỏ là segment "đang ngon" cho doanh nghiệp — nhiều người tương tác chuyển thành lead. Segment dưới đường đỏ là "đang yếu". Nếu phân khúc 2–4 triệu CVR cao nhất → đó là sweet spot của thị trường trọ. Phân khúc > 12 triệu CVR thấp đột biến → nghi vấn mis-categorize (Chart S1, S2).

**Hướng giải pháp:**
- *Quick-win*: ưu tiên đẩy tin ở sweet-spot segment lên top search.
- *Deep-dive*: với segment CVR thấp, drill-down xem có phải do tin sơ sài hay do data quality (sai cat).

### Chart D3 — CVR theo Top 10 quận

In [ ]:
# D3 — CVR theo district
con.register('dim_loc', df_dim[['item_id','district_name','city_name']])
sql = """
SELECT
  l.district_name, l.city_name,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
LEFT JOIN dim_loc l ON e.item_id = l.item_id
WHERE l.district_name IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) > 1000
ORDER BY n_events DESC
LIMIT 15
"""
df_d3 = con.execute(sql).df()

fig, ax = plt.subplots(figsize=(13, 6))
labels = [f"{r['district_name']} ({r['city_name']})" for _, r in df_d3.iterrows()]
ax.barh(labels[::-1], df_d3['cvr_pct'].values[::-1], color='cadetblue')
ax.set_title('CVR (%) — Top 15 Quận theo Số Events (lọc >1000 events)')
ax.set_xlabel('CVR (%)')
overall = df_d3['n_contact'].sum() / df_d3['n_events'].sum() * 100
ax.axvline(overall, color='red', linestyle='--', label=f'CVR top 15 = {overall:.2f}%')
ax.legend()
plt.tight_layout(); plt.show()
print(df_d3.round(2))

**Mô tả biểu đồ:** CVR của 15 quận có nhiều events nhất (lọc bỏ quận có < 1000 events để tránh nhiễu).

**Insight nghiệp vụ:** Quận có CVR cao = thị trường "khát phòng" — người dùng quyết định nhanh khi thấy tin. Quận CVR thấp = cung dư hoặc tin kém chất lượng. So sánh với Chart S4 (top supply) → quận top supply mà CVR thấp = dư cung. Quận top supply mà CVR cao = market hot.

**Hướng giải pháp:**
- *Quick-win*: tăng đầu tư marketing seller ở quận CVR cao (vì sản phẩm dễ "bán").
- *Deep-dive*: ở quận CVR thấp, drill-down xem nguyên nhân — quá đắt, ít ảnh, hay user nhầm địa lý?

### Chart D4 — CVR & Volume theo Event Type

In [ ]:
# D4 — Volume + CVR theo event_type (surface chỉ có 2 giá trị nên không vẽ).
sql = """
SELECT
  event_type,
  COUNT(*) AS n_events,
  SUM(is_contact) AS n_contact,
  SUM(is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final
GROUP BY 1
ORDER BY n_events DESC
"""
df_d4 = con.execute(sql).df()
print(df_d4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.pie(df_d4['n_events'], labels=df_d4['event_type'], autopct='%1.1f%%', startangle=90)
ax1.set_title('Phân bố Events theo Event Type')
ax2.bar(df_d4['event_type'], df_d4['cvr_pct'], color='salmon')
ax2.set_title('CVR (%) theo Event Type')
ax2.set_ylabel('CVR (%)')
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right')
for i, v in enumerate(df_d4['cvr_pct']):
    ax2.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Trái — tỷ trọng từng `event_type` trong tổng events. Phải — CVR ứng với mỗi event_type.

**Lưu ý:** `is_contact` được recompute ở Zone 2 (A.3) = 1 cho các event tích cực `view_phone / contact_chat / contact_zalo / contact_sms / other_interaction`. Vì vậy CVR của các event này = 100% theo định nghĩa. CVR của `pageview` = 0% theo định nghĩa. Đây là sanity check sau preprocessing.

**Insight nghiệp vụ:** `pageview` chiếm bao nhiêu % tổng events cho biết user "lướt nhiều" hay "tương tác nhiều". `other_interaction` chiếm phần lớn → user phòng trọ tương tác nhiều với listing (zoom ảnh, mở rộng mô tả). Tỷ lệ `view_phone` vs `contact_chat` cho biết hành vi chính của user là gọi hay nhắn.

**Hướng giải pháp:**
- *Quick-win*: tối ưu nút action chính theo event_type chiếm tỷ trọng cao (nếu view_phone > chat → đặt nút SĐT ưu tiên).
- *Deep-dive*: phân tích thêm `surface` field thực có nghĩa gì khi chỉ có 2 giá trị `ad_view`/`adview` — có thể đây là split web/app hay đo lường thử nghiệm.

### Chart D5 — Phân phối `dwell_time_sec` (contact vs không contact)

In [ ]:
# D5 — Dwell time distribution
sql = """
SELECT
  is_contact,
  QUANTILE_CONT(dwell_time_sec, 0.25) AS p25,
  QUANTILE_CONT(dwell_time_sec, 0.50) AS p50,
  QUANTILE_CONT(dwell_time_sec, 0.75) AS p75,
  QUANTILE_CONT(dwell_time_sec, 0.90) AS p90,
  AVG(dwell_time_sec) AS avg_d,
  COUNT(*) AS n
FROM ev_final
WHERE dwell_time_sec IS NOT NULL AND dwell_time_sec > 0
GROUP BY is_contact
"""
df_d5 = con.execute(sql).df()
print(df_d5.round(2))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(4)
width = 0.35
for i, lbl in enumerate(['Không contact', 'Có contact']):
    row = df_d5[df_d5['is_contact'] == i].iloc[0]
    vals = [row['p25'], row['p50'], row['p75'], row['p90']]
    ax.bar(x + (i - 0.5) * width, vals, width, label=lbl)
ax.set_xticks(x)
ax.set_xticklabels(['P25', 'P50', 'P75', 'P90'])
ax.set_title('Phân phối Dwell Time (giây) — Contact vs Không Contact')
ax.set_ylabel('Dwell time (s)')
ax.legend()
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Phân vị dwell time (P25/50/75/90) so sánh giữa events có vs không có contact.

**Insight nghiệp vụ:** User có contact thường dành nhiều thời gian xem tin hơn (P50 cao gấp 2-3 lần) — họ đọc kỹ rồi mới quyết định liên hệ. Nếu khoảng cách giữa 2 nhóm hẹp → tin chưa đủ hấp dẫn để giữ chân user.

**Hướng giải pháp:**
- *Quick-win*: thêm gallery ảnh dạng carousel, video tour ngắn để tăng dwell time.
- *Deep-dive*: dùng dwell time làm feature đầu vào cho model — user dwell > 30s ở 1 tin nên được recommend tin tương tự (collaborative signal).

### Chart D6 — Login vs Non-login (session-scoped)

In [ ]:
# D6 — Login vs non-login CVR
sql = """
SELECT
  is_login,
  COUNT(DISTINCT session_id) AS n_sessions,
  COUNT(*) AS n_events,
  SUM(is_contact) AS n_contact,
  SUM(is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final
GROUP BY is_login
"""
df_d6 = con.execute(sql).df()
print(df_d6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.bar(df_d6['is_login'], df_d6['n_sessions'], color=['#2ca02c','#d62728'])
ax1.set_title('Số Session theo Trạng thái Login')
ax1.set_ylabel('Số session')
for i, v in enumerate(df_d6['n_sessions']):
    ax1.text(i, v, f'{v:,}', ha='center', va='bottom')

ax2.bar(df_d6['is_login'], df_d6['cvr_pct'], color=['#2ca02c','#d62728'])
ax2.set_title('CVR (%) theo Trạng thái Login')
ax2.set_ylabel('CVR (%)')
for i, v in enumerate(df_d6['cvr_pct']):
    ax2.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Trái — số session login vs non-login. Phải — CVR của 2 nhóm.

**Insight nghiệp vụ:** Theo rule Cursor.md, **chỉ user login mới tạo ra doanh thu thực** (contact thực sự). Tỷ lệ non-login session cao là cơ hội để chuyển đổi — user chưa đăng nhập đang "lướt thử", nếu UX khuyến khích đăng nhập đúng thời điểm thì CVR tổng sẽ tăng.

**Hướng giải pháp:**
- *Quick-win*: hiển thị popup "Đăng nhập để xem SĐT" ở bước Intent (Chart D1) — chặn ở đúng moment-of-truth.
- *Deep-dive*: với user non-login, dùng session_id để build short-term recommendation; với user login, build long-term profile cho personalization.

## Zone 10 — Marketplace Dynamics: Cung ↔ Cầu

Theo tư duy "ông mai bà mối": platform chỉ kiếm tiền khi ghép đúng cung-cầu. 4 charts trong zone này phát hiện vùng dư cung / thiếu cung và độ hiệu quả ghép đôi.

### Chart M1 — Heatmap Cung × Cầu theo Quận × Phân khúc giá

In [ ]:
# M1 — Heatmap supply x demand
# Supply: số tin per (district, price_bin)
df_dim['price_bin'] = pd.cut(df_dim['price_mid_vnd'].fillna(-1) / 1e6,
    bins=[-2, 0, 2, 4, 6, 8, 12, 100],
    labels=['(NA)','<2tr','2-4tr','4-6tr','6-8tr','8-12tr','>12tr'])

supply_grid = (df_dim.groupby(['district_name','price_bin'])
                     .size().unstack(fill_value=0))
# Lấy top 12 district có nhiều tin nhất
top_dist = df_dim['district_name'].value_counts().head(12).index
supply_grid = supply_grid.loc[supply_grid.index.intersection(top_dist)]

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(supply_grid, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            cbar_kws={'label':'Số tin (Supply)'})
ax.set_title('Heatmap Cung — Top 12 Quận × Phân khúc Giá')
ax.set_xlabel('Phân khúc giá')
ax.set_ylabel('Quận')
plt.tight_layout(); plt.show()
print('\nSupply matrix shape:', supply_grid.shape)

**Mô tả biểu đồ:** Số lượng tin đăng (cung) phân bố theo top 12 quận × 7 phân khúc giá.

**Insight nghiệp vụ:** Ô đậm = cung dồi dào. Nếu một ô đậm nhưng CVR thấp ở Chart D2/D3 thì đó là vùng **dư cung**, cần marketing để kích cầu hoặc khuyến khích seller giảm giá. Ô nhạt ở quận thuộc top demand (Bình Tân, Tân Bình, Cầu Giấy...) là vùng **thiếu cung** — cơ hội cho marketing seller mới.

**Hướng giải pháp:**
- *Quick-win*: với ô thiếu cung, đẩy thông báo "Quận X đang khan tin" cho seller để khuyến khích đăng.
- *Deep-dive*: phân tích sao một số quận có cung quá tập trung ở 1 phân khúc giá — có thể do đặc điểm địa lý (KCN, sinh viên).

### Chart M2 — Supply-Demand Gap theo Quận

In [ ]:
# M2 — Gap = log(supply / demand_events)
con.register('dim_loc2', df_dim[['item_id','district_name']])
sql = """
SELECT
  l.district_name,
  COUNT(DISTINCT e.session_id) AS demand_sessions,
  COUNT(*) AS demand_events
FROM ev_final e
LEFT JOIN dim_loc2 l ON e.item_id = l.item_id
WHERE l.district_name IS NOT NULL
GROUP BY 1
"""
df_demand = con.execute(sql).df()
df_supply = df_dim.groupby('district_name').size().reset_index(name='supply_items')

df_m2 = df_supply.merge(df_demand, on='district_name', how='inner')
df_m2['ratio'] = df_m2['supply_items'] / df_m2['demand_sessions'].clip(lower=1)
df_m2['log_ratio'] = np.log(df_m2['ratio'].replace(0, np.nan))
df_m2 = df_m2[df_m2['supply_items'] >= 30].sort_values('log_ratio')
top_bottom = pd.concat([df_m2.head(10), df_m2.tail(10)])

fig, ax = plt.subplots(figsize=(13, 7))
colors = ['#d62728' if v > 0 else '#2ca02c' for v in top_bottom['log_ratio']]
ax.barh(top_bottom['district_name'], top_bottom['log_ratio'], color=colors)
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Supply-Demand Gap = log(Supply / Demand sessions) — Top thiếu (xanh) & dư (đỏ)')
ax.set_xlabel('log ratio (negative = thiếu cung, positive = dư cung)')
plt.tight_layout(); plt.show()
print(df_m2[['district_name','supply_items','demand_sessions','log_ratio']].round(3).head(10))

**Mô tả biểu đồ:** Quận có gap âm (xanh) = **thiếu cung** (cầu vượt cung). Gap dương (đỏ) = **dư cung**.

**Insight nghiệp vụ:** Đây là "danh sách hành động" cho team marketplace:
- Quận **thiếu cung** → recruit seller mới, ưu đãi đăng tin miễn phí.
- Quận **dư cung** → cần marketing cho demand (quảng cáo người tìm trọ).

**Hướng giải pháp:**
- *Quick-win*: dashboard tuần cho biz team xem 5 quận thiếu cung nhất để target seller.
- *Deep-dive*: nghiên cứu pattern theo thời gian — gap có theo mùa (sinh viên đầu năm, công nhân về Tết)?

### Chart M3 — Match efficiency (% tin có ≥ 1 contact trong 7 ngày đầu)

In [ ]:
# M3 — Match efficiency từ snapshot
sql = """
WITH first_7d AS (
  SELECT item_id, MAX(contacts_24h_capped) AS max_contact_7d
  FROM snap_clean
  WHERE listing_age_days <= 7
  GROUP BY item_id
)
SELECT
  CASE WHEN max_contact_7d > 0 THEN 'Có contact' ELSE 'Không contact' END AS matched,
  COUNT(*) AS n
FROM first_7d
GROUP BY 1
"""
df_m3 = con.execute(sql).df()
print(df_m3)

# Breakdown theo price_bin
con.register('dim_pb', df_dim[['item_id','price_bin']])
sql2 = """
WITH first_7d AS (
  SELECT item_id, MAX(contacts_24h_capped) AS max_contact_7d
  FROM snap_clean
  WHERE listing_age_days <= 7
  GROUP BY item_id
)
SELECT
  COALESCE(d.price_bin, '(NA)') AS price_bin,
  COUNT(*) AS n_total,
  SUM(CASE WHEN f.max_contact_7d > 0 THEN 1 ELSE 0 END) AS n_matched,
  SUM(CASE WHEN f.max_contact_7d > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS match_pct
FROM first_7d f
LEFT JOIN dim_pb d ON f.item_id = d.item_id
GROUP BY 1
ORDER BY match_pct DESC
"""
df_m3b = con.execute(sql2).df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_m3b['price_bin'], df_m3b['match_pct'], color='mediumseagreen')
ax.set_title('Match Efficiency: % tin có Contact trong 7 ngày đầu — theo Phân khúc Giá')
ax.set_ylabel('Match %')
overall = df_m3b['n_matched'].sum() / df_m3b['n_total'].sum() * 100
ax.axhline(overall, color='red', linestyle='--', label=f'Match tổng = {overall:.2f}%')
for i, v in enumerate(df_m3b['match_pct']):
    ax.text(i, v, f'{v:.1f}%', ha='center', va='bottom')
ax.legend()
plt.tight_layout(); plt.show()
print(df_m3b.round(2))

**Mô tả biểu đồ:** % tin đăng có ít nhất 1 contact trong 7 ngày đầu, breakdown theo phân khúc giá.

**Insight nghiệp vụ:** Đây là chỉ số "hiệu quả mai mối" — tỷ lệ ghép đôi thành công ở giai đoạn vàng. Nếu < 30% trong 7 ngày đầu → có vấn đề nghiêm trọng với recommender hoặc chất lượng tin. Phân khúc match cao thường là sweet spot — phù hợp với cả supply và demand.

**Hướng giải pháp:**
- *Quick-win*: tin chưa match trong 3 ngày đầu nên được boost lên top search (auto-refresh).
- *Deep-dive*: drill-down "dead listings" (Chart M4) để xem lý do match thấp.

### Chart M4 — Dead listings profile (0 contact sau 14 ngày)

In [ ]:
# M4 — Dead listings: 3 bar charts (district, price, furnishing)
sql = """
WITH first_14d AS (
  SELECT item_id, MAX(contacts_24h_capped) AS max_contact_14d
  FROM snap_clean
  WHERE listing_age_days <= 14
  GROUP BY item_id
)
SELECT item_id FROM first_14d WHERE max_contact_14d = 0
"""
dead_items = con.execute(sql).df()
dead_set = set(dead_items['item_id'].tolist())
df_dim['_is_dead'] = df_dim['item_id'].isin(dead_set)

print(f'Dead listings: {df_dim["_is_dead"].sum():,} / {len(df_dim):,} ({df_dim["_is_dead"].mean()*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# By district top 10
dead_by_dist = df_dim.groupby('district_name')['_is_dead'].mean().sort_values(ascending=False).head(10) * 100
axes[0].barh(dead_by_dist.index[::-1], dead_by_dist.values[::-1], color='crimson')
axes[0].set_title('% Dead Listings — Top 10 Quận (cao nhất)')
axes[0].set_xlabel('% dead')

# By price_bin
dead_by_price = df_dim.groupby('price_bin', observed=True)['_is_dead'].mean() * 100
axes[1].bar(dead_by_price.index.astype(str), dead_by_price.values, color='darkorange')
axes[1].set_title('% Dead Listings theo Phân khúc Giá')
axes[1].set_ylabel('% dead')
for i, v in enumerate(dead_by_price.values):
    axes[1].text(i, v, f'{v:.1f}%', ha='center', va='bottom')

# By furnishing
dead_by_furn = df_dim.fillna({'furnishing':'(NA)'}).groupby('furnishing')['_is_dead'].mean().sort_values(ascending=False).head(8) * 100
axes[2].barh(dead_by_furn.index[::-1], dead_by_furn.values[::-1], color='purple')
axes[2].set_title('% Dead Listings theo Nội thất')
axes[2].set_xlabel('% dead')

plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** 3 góc nhìn về "dead listings" (tin 0 contact sau 14 ngày): theo quận / giá / nội thất.

**Insight nghiệp vụ:** Dead listings là rác trong feed — chiếm tài nguyên hiển thị mà không tạo doanh thu. Nếu top quận dead trùng với top quận supply (Chart S4) → quận đó cần intervention. Phân khúc giá > 12tr dead nhiều = nghi vấn data quality (Chart S1, S2). Nội thất "(NA)" / nhà trống dead nhiều = thiếu thông tin khiến user không quyết định.

**Hướng giải pháp:**
- *Quick-win*: auto-hide tin 0 contact sau 21 ngày, gửi notification cho seller "Tin chưa hiệu quả — cập nhật giá/ảnh".
- *Deep-dive*: A/B test cho seller dead-listing: gợi ý giảm giá 5–10% và xem tỷ lệ revive.

## Zone 11 — Empathy Feature Engineering Hints

Đặt mình vào vị trí **người thuê trọ** — họ quan tâm gì? 7 chart sau là các feature ứng viên cho recommender LightGBM, validate bằng CVR.

### Chart E1 — Agent vs Private CVR

In [ ]:
# E1 — seller_type x CVR
con.register('dim_st', df_dim[['item_id','seller_type']])
sql = """
SELECT
  COALESCE(d.seller_type, '(NA)') AS seller_type,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
LEFT JOIN dim_st d ON e.item_id = d.item_id
GROUP BY 1
ORDER BY n_events DESC
"""
df_e1 = con.execute(sql).df()
print(df_e1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_e1['seller_type'], df_e1['cvr_pct'], color=['#1f77b4','#ff7f0e','#888'])
ax.set_title('CVR (%) — Agent vs Private (Chủ trọ)')
ax.set_ylabel('CVR (%)')
for i, v in enumerate(df_e1['cvr_pct']):
    ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** CVR so sánh giữa tin của Agent (môi giới) và Private (chủ trọ cá nhân).

**Insight nghiệp vụ:** Trong phòng trọ, người thuê thường ưu ái **private** vì tránh phí môi giới + giao tiếp trực tiếp dễ thương lượng. Nếu private CVR > agent → giả thuyết được khẳng định, recommender nên ưu tiên private khi user mới đến platform.

**Hướng giải pháp:**
- *Quick-win*: thêm badge "Chủ trọ" cho private listings, hiển thị nổi bật trong card.
- *Deep-dive*: nghiên cứu xem một số agent có CVR cao bất thường (thương hiệu uy tín?) để học pattern.

### Chart E2 — Furnishing × CVR

In [ ]:
# E2 — furnishing x CVR
con.register('dim_furn', df_dim[['item_id','furnishing']])
sql = """
SELECT
  COALESCE(d.furnishing, '(không khai)') AS furnishing,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
LEFT JOIN dim_furn d ON e.item_id = d.item_id
GROUP BY 1
HAVING COUNT(*) > 500
ORDER BY cvr_pct DESC
"""
df_e2 = con.execute(sql).df()
print(df_e2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_e2['furnishing'], df_e2['cvr_pct'], color=sns.color_palette('viridis', len(df_e2)))
ax.set_title('CVR (%) theo Tình trạng Nội thất')
ax.set_ylabel('CVR (%)')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(df_e2['cvr_pct']):
    ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** CVR phân theo từng loại nội thất (lọc các nhóm > 500 events).

**Insight nghiệp vụ:** Người thuê trọ thường có 2 nhóm chính:
- **Sinh viên / công nhân**: cần nội thất đầy đủ (giường, tủ, máy lạnh) để dọn vào liền.
- **Gia đình / người chuyển công tác**: thường mang đồ riêng, thích nhà trống/cơ bản.

Nếu CVR "Nội thất đầy đủ" cao nhất → segment chính là user "lười" cần dọn vào ngay. Tin "(không khai)" CVR thấp = thiếu info dẫn đến user không quyết định.

**Hướng giải pháp:**
- *Quick-win*: bắt buộc nhập furnishing khi đăng tin.
- *Deep-dive*: build 2 segment recommender: "user prefer full-furnished" vs "user prefer empty" dựa trên hành vi click.

### Chart E3 — Seller Communication Style (đã sửa proxy)

In [ ]:
# E3.a — Response density = chat_messages / chat_turns per seller
con.register('dim_seller', df_dim[['item_id','seller_id']])
sql_density = """
SELECT
  d.seller_id,
  SUM(c.chat_message_count) AS total_msg,
  SUM(c.chat_turn_count) AS total_turn,
  SUM(c.chat_message_count) * 1.0 / NULLIF(SUM(c.chat_turn_count), 0) AS msg_per_turn,
  COUNT(DISTINCT d.item_id) AS n_listings
FROM contact_clean c
INNER JOIN dim_seller d ON c.item_id = d.item_id
GROUP BY d.seller_id
HAVING SUM(c.chat_turn_count) >= 5
ORDER BY total_turn DESC
LIMIT 500
"""
df_e3a = con.execute(sql_density).df()
print(f'Sellers có >= 5 turn chat: {len(df_e3a)}')
print(df_e3a['msg_per_turn'].describe().round(2))

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_e3a['msg_per_turn'].dropna(), bins=30, color='steelblue', edgecolor='white')
ax.axvline(df_e3a['msg_per_turn'].median(), color='red', linestyle='--',
           label=f'Trung vị = {df_e3a["msg_per_turn"].median():.2f} msg/turn')
ax.set_title('E3.a — Response Density: Tin nhắn trên mỗi lượt qua lại (per seller, ≥ 5 turn)')
ax.set_xlabel('msg / turn')
ax.set_ylabel('Số seller')
ax.legend()
plt.tight_layout(); plt.show()

**Mô tả biểu đồ E3.a:** Phân phối "tin nhắn trung bình trên mỗi lượt chat qua lại" — đo seller "rep dài" hay "rep ngắn".

**Insight nghiệp vụ:** Seller msg/turn cao (rep dài, tư vấn nhiều) thường có CVR cao hơn vì user cảm giác được chăm sóc. Seller msg/turn thấp (rep cụt ngủn) khiến user mất hứng.

**Hướng giải pháp:**
- *Quick-win*: tooltip cho seller "Câu trả lời quá ngắn, hãy bổ sung địa chỉ/giờ xem nhà".
- *Deep-dive*: train chatbot suggest reply để giúp seller mới có chất lượng giao tiếp đồng đều.

In [ ]:
# E3.b — Communication preference: view_phone / (view_phone + chat) per seller
sql_pref = """
SELECT
  d.seller_id,
  SUM(CASE WHEN e.event_type = 'view_phone' THEN 1 ELSE 0 END) AS n_view_phone,
  SUM(CASE WHEN e.event_type = 'contact_chat' THEN 1 ELSE 0 END) AS n_chat,
  SUM(CASE WHEN e.event_type = 'view_phone' THEN 1 ELSE 0 END) * 1.0
    / NULLIF(SUM(CASE WHEN e.event_type IN ('view_phone','contact_chat') THEN 1 ELSE 0 END), 0) AS phone_ratio
FROM ev_final e
INNER JOIN dim_seller d ON e.item_id = d.item_id
GROUP BY d.seller_id
HAVING SUM(CASE WHEN e.event_type IN ('view_phone','contact_chat') THEN 1 ELSE 0 END) >= 10
"""
df_e3b = con.execute(sql_pref).df()
print(f'Sellers có >= 10 intent events: {len(df_e3b)}')

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_e3b['phone_ratio'].dropna(), bins=30, color='darkorange', edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', label='= 0.5 (cân bằng)')
ax.set_title('E3.b — Communication Preference: phone_ratio = view_phone / (view_phone + chat)')
ax.set_xlabel('Phone ratio (0 = chat-oriented, 1 = phone-oriented)')
ax.set_ylabel('Số seller')
ax.legend()
plt.tight_layout(); plt.show()
print('Phân vị:', df_e3b['phone_ratio'].quantile([.25,.5,.75]).round(3).to_dict())

**Mô tả biểu đồ E3.b:** Tỷ lệ user xem SĐT vs mở chat của seller — đo seller "thiên hướng gọi" hay "thiên hướng chat".

**Insight nghiệp vụ:** Seller có phone_ratio cao (> 0.7) → user toàn xem SĐT rồi gọi điện, ít chat trên platform → platform khó tracking outcome. Seller phone_ratio thấp (< 0.3) → user thích chat, dễ tracking nhưng tốn thời gian. Phân khúc trọ thường lệch về phone_ratio cao vì người thuê muốn quyết định nhanh.

**Hướng giải pháp:**
- *Quick-win*: với seller phone-heavy, encourage "chat trước khi gọi" để có log trên platform.
- *Deep-dive*: dùng phone_ratio làm feature, kết hợp với purchased để estimate true conversion rate (phone calls thường có CVR cao hơn nhưng không log).

### Chart E4 — Số bathrooms × CVR (chỉ nếu null < 70%)

In [ ]:
# E4 — bathrooms x CVR
if 'bathrooms' in df_dim.columns:
    null_rate = df_dim['bathrooms'].isna().mean()
    print(f'bathrooms null rate: {null_rate*100:.1f}%')
    if null_rate < 0.70:
        con.register('dim_bath', df_dim[['item_id','bathrooms']])
        sql = """
        SELECT
          COALESCE(CAST(d.bathrooms AS INTEGER), -1) AS n_bath,
          COUNT(*) AS n_events,
          SUM(e.is_contact) AS n_contact,
          SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
        FROM ev_final e
        LEFT JOIN dim_bath d ON e.item_id = d.item_id
        WHERE d.bathrooms IS NOT NULL
        GROUP BY 1 ORDER BY 1
        """
        df_e4 = con.execute(sql).df()
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(df_e4['n_bath'].astype(str), df_e4['cvr_pct'], color='teal')
        ax.set_title('CVR (%) theo Số phòng tắm')
        ax.set_ylabel('CVR (%)')
        for i, v in enumerate(df_e4['cvr_pct']):
            ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
        plt.tight_layout(); plt.show()
        print(df_e4)
    else:
        print('Bathrooms null rate quá cao — skip chart E4')
else:
    print('Column bathrooms đã bị drop ở Zone 4 C.4 — skip chart E4')

**Mô tả biểu đồ:** CVR theo số bathrooms (chỉ chạy nếu cột không null quá nhiều).

**Insight nghiệp vụ:** Phòng trọ thông thường có 1 WC chung. Phòng có WC riêng (≥ 1) thường giá cao hơn nhưng CVR cũng cao hơn vì độ riêng tư. Nếu bathrooms null > 70% → cột không đáng tin để làm feature.

**Hướng giải pháp:**
- *Quick-win*: nếu cột bị thiếu nhiều, parse từ `title` keyword "WC riêng" để derive bathrooms.
- *Deep-dive*: A/B test segment "có WC riêng" cho user trẻ (sinh viên / nhân viên văn phòng) để tăng CVR.

### Chart E5 — `images_count` bins × CVR

In [ ]:
# E5 — images_count x CVR
con.register('dim_img', df_dim[['item_id','images_count']])
sql = """
SELECT
  CASE
    WHEN d.images_count IS NULL OR d.images_count = 0 THEN '0 ảnh'
    WHEN d.images_count <= 2 THEN '1-2 ảnh'
    WHEN d.images_count <= 5 THEN '3-5 ảnh'
    WHEN d.images_count <= 10 THEN '6-10 ảnh'
    ELSE '>10 ảnh'
  END AS img_bin,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
LEFT JOIN dim_img d ON e.item_id = d.item_id
GROUP BY 1
ORDER BY MIN(d.images_count) NULLS FIRST
"""
df_e5 = con.execute(sql).df()

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(df_e5['img_bin'], df_e5['cvr_pct'], color='goldenrod')
ax.set_title('CVR (%) theo Số ảnh trong Tin')
ax.set_ylabel('CVR (%)')
for i, v in enumerate(df_e5['cvr_pct']):
    ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
plt.tight_layout(); plt.show()
print(df_e5)

**Mô tả biểu đồ:** CVR phân theo số ảnh trên tin (5 bins).

**Insight nghiệp vụ:** Thường CVR tăng dần theo số ảnh, plateau ở ~6-10 ảnh. Tin 0 ảnh có CVR cực thấp → "rác". Sau ngưỡng 10 ảnh, CVR có thể giảm vì over-information làm user choáng.

**Hướng giải pháp:**
- *Quick-win*: gợi ý "Tin có ít nhất 5 ảnh nhận được lead cao hơn 2x — bổ sung ngay" cho seller.
- *Deep-dive*: phân tích chất lượng ảnh (resolution, độ sáng) — không phải nhiều ảnh là tốt mà phải đẹp.

### Chart E6 — Keyword trong tiêu đề × CVR

In [ ]:
# E6 — Title keyword presence (cần normalize tiếng Việt)
def normalize_vn(s):
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.lower()

df_dim['title_norm'] = df_dim['title'].apply(normalize_vn)

# Map: slug ASCII (cho SQL col name) → (label hiển thị, patterns)
KEYWORDS = {
    'maylanh':   ('Máy lạnh',        ['may lanh', 'mlanh', 'dieu hoa']),
    'wifi':      ('Wifi',            ['wifi']),
    'bancong':   ('Ban công',        ['ban cong', 'bancong']),
    'gan':       ('Gần (trường/chợ)',['gan truong', 'gan cho', 'gan cv', 'gan trung tam', 'gan dh', 'gan kcn']),
    'trungtam':  ('Trung tâm',       ['trung tam']),
    'moi':       ('Mới',             ['moi xay', 'moi build', ' moi ', 'newly']),
    'giare':     ('Giá rẻ',          ['gia re', 'gia tot']),
    'antoan':    ('An ninh / sạch',  ['an ninh', 'sach se', 'thoang mat']),
    'fullnt':    ('Full nội thất',   ['full nt', 'full noi that', 'day du noi that']),
}

# Tạo các cột _kw_<slug> trên df_dim
for slug, (label, patterns) in KEYWORDS.items():
    pat = '|'.join(re.escape(p) for p in patterns)
    df_dim[f'_kw_{slug}'] = df_dim['title_norm'].str.contains(pat, regex=True, na=False)

kw_cols = [f'_kw_{slug}' for slug in KEYWORDS]
con.register('dim_kw', df_dim[['item_id'] + kw_cols])

results = []
for slug, (label, _) in KEYWORDS.items():
    col = f'_kw_{slug}'  # ASCII column → SQL safe
    sql = f"""
    SELECT
      d.{col} AS has_kw,
      COUNT(*) AS n_events,
      SUM(e.is_contact) AS n_contact,
      SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
    FROM ev_final e
    LEFT JOIN dim_kw d ON e.item_id = d.item_id
    WHERE d.{col} IS NOT NULL
    GROUP BY 1
    """
    df_r = con.execute(sql).df()
    if len(df_r) >= 2:
        try:
            cvr_yes = float(df_r[df_r['has_kw'] == True]['cvr_pct'].iloc[0])
            cvr_no  = float(df_r[df_r['has_kw'] == False]['cvr_pct'].iloc[0])
            n_yes = int(df_r[df_r['has_kw'] == True]['n_events'].iloc[0])
            results.append({'keyword': label, 'cvr_yes': cvr_yes, 'cvr_no': cvr_no,
                            'lift': cvr_yes - cvr_no, 'n_events_yes': n_yes})
        except (IndexError, KeyError):
            pass

df_e6 = pd.DataFrame(results)
if len(df_e6) == 0:
    print('⚠️ Không có keyword nào có data — bỏ qua chart E6')
else:
    df_e6 = df_e6.sort_values('lift', ascending=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(df_e6['keyword'], df_e6['lift'],
            color=['#2ca02c' if v > 0 else '#d62728' for v in df_e6['lift']])
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title('Lift CVR (%) khi Title chứa Keyword (vs không chứa)')
    ax.set_xlabel('Δ CVR (percentage point)')
    plt.tight_layout(); plt.show()
    print(df_e6.round(3))

**Mô tả biểu đồ:** Sự khác biệt CVR (Δ) giữa tin CÓ vs KHÔNG có keyword trong title.

**Insight nghiệp vụ:** Keyword Δ > 0 = keyword "bán được hàng" (user thấy là click). Δ < 0 = keyword "phản tác dụng" (có thể là tin spam/clickbait). "Máy lạnh", "wifi", "ban công" thường lift dương. "Giá rẻ" có thể lift âm vì user nghi ngờ.

**Hướng giải pháp:**
- *Quick-win*: gợi ý keyword khi seller đăng tin — "Tin có 'máy lạnh' nhận lead nhiều hơn 30%".
- *Deep-dive*: build NLP model phân loại "tin tốt" vs "tin clickbait" dựa trên title + image + price.

### Chart E7 — Value-for-money: CVR theo `price_per_sqm` × Quận

In [ ]:
# E7 — CVR theo price_per_sqm bins, breakdown theo top 5 district
df_dim['pps_bin'] = pd.cut(df_dim['price_per_sqm'].fillna(-1) / 1e3,
    bins=[-1, 100, 200, 300, 500, 1000, 5000],
    labels=['< 100K','100-200K','200-300K','300-500K','500K-1tr','>1tr'])

top5_dist = df_dim['district_name'].value_counts().head(5).index.tolist()
con.register('dim_e7', df_dim[df_dim['district_name'].isin(top5_dist)][['item_id','district_name','pps_bin']])

sql = """
SELECT
  d.district_name,
  d.pps_bin,
  COUNT(*) AS n_events,
  SUM(e.is_contact) AS n_contact,
  SUM(e.is_contact) * 100.0 / COUNT(*) AS cvr_pct
FROM ev_final e
INNER JOIN dim_e7 d ON e.item_id = d.item_id
WHERE d.pps_bin IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) > 200
"""
df_e7 = con.execute(sql).df()
piv = df_e7.pivot(index='district_name', columns='pps_bin', values='cvr_pct')

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(piv, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            cbar_kws={'label':'CVR (%)'})
ax.set_title('Heatmap CVR (%) — Top 5 Quận × Giá / m² (nghìn đồng)')
ax.set_xlabel('Giá / m²')
ax.set_ylabel('Quận')
plt.tight_layout(); plt.show()
print(piv.round(2))

**Mô tả biểu đồ:** CVR theo từng bin giá/m² × top 5 quận. Ô xanh = CVR cao, đỏ = thấp.

**Insight nghiệp vụ:** Mỗi quận có một **"ngưỡng giá/m² hợp lý"** — vùng xanh là ngưỡng mà người thuê chấp nhận. Ô đỏ ở giá/m² thấp → có thể tin ảo. Ô đỏ ở giá/m² cao → đắt quá so với mặt bằng. Recommender nên ưu tiên tin nằm trong vùng xanh của quận user đang xem.

**Hướng giải pháp:**
- *Quick-win*: hiển thị label "Giá hợp lý cho khu vực" trên tin nằm trong vùng xanh local district.
- *Deep-dive*: dùng `price_per_sqm` percentile-per-district làm feature cho LightGBM model (rank within district).

## Zone 11 — Empathy Feature Engineering
TBD

## Zone 12 — Problem-driven Actionable Insights (tổng kết)

Tổng hợp các vấn đề phát hiện qua Zone 8-11, rank theo impact, đưa ra plan hành động cụ thể.

### Chart P1 — Top 5 vấn đề lớn nhất (rank by impact)

In [ ]:
# P1 — Rank impact = |CVR_loss| × volume_affected
# Tính CVR tổng cat 1050
overall_cvr = con.execute("""
SELECT SUM(is_contact)*100.0/COUNT(*) FROM ev_final
""").fetchone()[0]
print(f'CVR tổng cat 1050: {overall_cvr:.3f}%')

issues = []

# Issue 1: Tin 0 ảnh
zero_img = df_dim[df_dim['images_count'].fillna(0) == 0]
n_zero_img = len(zero_img)
issues.append({
    'issue': 'Tin 0 ảnh',
    'n_affected': n_zero_img,
    'pct_listings': n_zero_img / len(df_dim) * 100,
    'note': 'Thiếu hấp dẫn visual — CVR rất thấp'
})

# Issue 2: Dead listings (đã tính ở Zone 10 M4)
if '_is_dead' in df_dim.columns:
    n_dead = int(df_dim['_is_dead'].sum())
    issues.append({
        'issue': 'Dead listings (0 contact / 14 ngày)',
        'n_affected': n_dead,
        'pct_listings': n_dead / len(df_dim) * 100,
        'note': 'Rác trong feed, lãng phí tài nguyên hiển thị'
    })

# Issue 3: Tin "(không khai)" nội thất
n_no_furn = df_dim['furnishing'].isna().sum()
issues.append({
    'issue': 'Tin không khai nội thất',
    'n_affected': int(n_no_furn),
    'pct_listings': n_no_furn / len(df_dim) * 100,
    'note': 'Thiếu thông tin chính khiến user không quyết định'
})

# Issue 4: Tin nghi mis-categorize (area > 50 m²)
n_big = (df_dim['area_sqm'] > 50).sum()
issues.append({
    'issue': 'Diện tích > 50 m² (nghi nhà nguyên căn)',
    'n_affected': int(n_big),
    'pct_listings': n_big / len(df_dim) * 100,
    'note': 'Có thể bị mis-categorize, làm loãng search phòng trọ'
})

# Issue 5: Tin price_per_sqm cực thấp (< P5) — nghi tin ảo
if 'price_per_sqm' in df_dim.columns:
    p5 = df_dim['price_per_sqm'].quantile(0.05)
    n_fake = (df_dim['price_per_sqm'] < p5).sum() if pd.notna(p5) else 0
    issues.append({
        'issue': 'price/m² < P5 (nghi tin ảo)',
        'n_affected': int(n_fake),
        'pct_listings': n_fake / len(df_dim) * 100,
        'note': 'Giá quá hời thường là tin ảo / câu click'
    })

df_p1 = pd.DataFrame(issues).sort_values('n_affected', ascending=False)
print(df_p1)

fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(df_p1['issue'][::-1], df_p1['n_affected'][::-1], color='indianred')
ax.set_title('Top vấn đề Data/Listing — Cat 1050 — Số tin bị ảnh hưởng')
ax.set_xlabel('Số tin')
for i, (idx, row) in enumerate(df_p1[::-1].iterrows()):
    ax.text(row['n_affected'], i, f' {row["pct_listings"]:.1f}%', va='center')
plt.tight_layout(); plt.show()

**Mô tả biểu đồ:** Top vấn đề trên data cat 1050, rank theo số tin bị ảnh hưởng.

**Insight nghiệp vụ:** Đây là "to-do list" cho team marketplace. Vấn đề nào ảnh hưởng > 15% listings là **critical**, phải ưu tiên. Vấn đề nào < 5% là **nice-to-fix**, có thể để sau.

**Hướng giải pháp:** xem Chart P2 để chi tiết từng issue.

### Chart P2 — Bảng Quick-win vs Deep-dive cho mỗi vấn đề

In [ ]:
# P2 — Bảng giải pháp
solutions = [
    {'issue': 'Tin 0 ảnh',
     'quick_win': 'UX: bắt buộc tối thiểu 3 ảnh khi đăng tin',
     'deep_dive': 'Nghiên cứu seller pattern: là người mới (cần onboarding) hay người cũ (cần nhắc nhở)?'},
    {'issue': 'Dead listings (0 contact / 14 ngày)',
     'quick_win': 'Auto-hide tin > 21 ngày 0 contact, gửi notification "tin chưa hiệu quả"',
     'deep_dive': 'A/B test giảm giá 5-10% xem tỷ lệ revive là bao nhiêu'},
    {'issue': 'Tin không khai nội thất',
     'quick_win': 'Required field khi đăng tin',
     'deep_dive': 'Build NLP model parse furnishing từ title/description'},
    {'issue': 'Diện tích > 50 m² (nghi nhà nguyên căn)',
     'quick_win': 'Hard validation: area > 40 m² thì gợi ý chuyển cat 1020/1030',
     'deep_dive': 'Nghiên cứu hành vi seller cố tình sai cat để leo top — anti-gaming'},
    {'issue': 'price/m² < P5 (nghi tin ảo)',
     'quick_win': 'Cảnh báo "Giá quá thấp so với khu vực" + yêu cầu xác minh',
     'deep_dive': 'Build fraud detection model dùng price_per_sqm + image content'},
]
df_p2 = pd.DataFrame(solutions)
pd.set_option('display.max_colwidth', 100)
print(df_p2.to_string(index=False))

**Bảng giải pháp cho từng vấn đề:**

| Vấn đề | Quick-win (vá lỗi) | Deep-dive (root cause) |
|---|---|---|
| Tin 0 ảnh | Bắt buộc ≥ 3 ảnh khi đăng | Nghiên cứu seller pattern (mới vs cũ) |
| Dead listings | Auto-hide + notification | A/B test giảm giá 5-10% |
| Tin không khai nội thất | Required field | NLP parse từ title |
| Area > 50 m² | Hard validation, gợi ý chuyển cat | Anti-gaming research |
| price/m² < P5 | Cảnh báo + yêu cầu xác minh | Fraud detection model |

**Cách dùng**: business team chọn quick-win cho sprint hiện tại (1-2 tuần), giao deep-dive cho data team làm research dài hơi (1-2 tháng).

### Scorecard cuối cùng

In [ ]:
# Scorecard: so sánh CVR cat 1050 với benchmark 5 cat (từ file có sẵn)
bench_path = os.path.join(OUT_DIR, 's2_cvr.csv')
if os.path.exists(bench_path):
    df_bench = pd.read_csv(bench_path)
    print('Benchmark 5 cat (s2_cvr.csv):')
    print(df_bench)
else:
    df_bench = None
    print(f'⚠️ Không tìm thấy {bench_path} — bỏ qua benchmark.')

print(f'\n=== Scorecard Cat 1050 (Phòng trọ) ===')
print(f'• Tổng CVR cat 1050:        {overall_cvr:.3f}%')
print(f'• Số listings (sau clean):  {len(df_dim):,}')
print(f'• Số events (sau clean):    {con.execute("SELECT COUNT(*) FROM ev_final").fetchone()[0]:,}')
print(f'• Số issues phát hiện:      {len(df_quality)}')

# 3 segment cao nhất & thấp nhất (từ df_d2 — bins giá)
print('\n--- Segment giá CVR cao nhất ---')
print(df_d2.sort_values('cvr_pct', ascending=False).head(3)[['price_bin','cvr_pct']].to_string(index=False))
print('\n--- Segment giá CVR thấp nhất ---')
print(df_d2.sort_values('cvr_pct').head(3)[['price_bin','cvr_pct']].to_string(index=False))

### Kết luận

Notebook này hoàn thành 2 giai đoạn:

**Giai đoạn 1 (Zone 0-7) — Preprocessing**
- Bao phủ đủ 5 nhóm issue A-E của Cursor.md.
- Export `cat1050_v3_quality_report.csv` để modeling team tham chiếu.
- Tạo các cột phái sinh quan trọng: `price_low_vnd`, `price_high_vnd`, `price_mid_vnd`, `is_negotiable`, **`price_per_sqm`**, `user_scope_id`.

**Giai đoạn 2 (Zone 8-12) — EDA + Insight**
- 7 chart Supply, 6 chart Demand, 4 chart Marketplace Dynamics, 7 chart Empathy FE, 2 chart Insights.
- Mỗi chart có 3 phần: Mô tả + Insight + 2 hướng giải pháp (Quick-win + Deep-dive).
- Tổng kết Top 5 vấn đề + bảng plan hành động.

**Khuyến nghị bước tiếp theo:**
1. Dùng `cat1050_v3_quality_report.csv` để align với 4 cat còn lại (1010, 1020, 1030, 1040) — nhân rộng preprocess pattern.
2. Feature engineering cho LightGBM: `price_per_sqm`, `furnishing_clean`, `is_agent`, `images_count_bin`, `title_kw_*`, `seller_phone_ratio`, `seller_msg_per_turn`, `district_demand_pressure`.
3. Test các Quick-win với business team trước khi đầu tư Deep-dive.